# 环节 04 · 五个不可比陷阱（配套 Notebook）

> 配套：[benchmark.md](./benchmark.md) §二（全文最重要的一节）
> 导航：[环节00](./环节00-总揽与环节导航.md)
> 定位：把五条「数字看起来能比、其实不能比」落成对照表。数字全部来自 2026-09-12 本机实测（M4 Pro / 48GB）。**纯标准库。**

跑完应能回答：给我两行 tok/s，怎样判定它们有没有资格放进同一张表。


## 陷阱 1：拿错指标 —— prefill 与 decode 是两种负载


In [ ]:
# 同模型 MLX Qwen3-0.6B，benchmark §五 / §七
prefill = {"bf16": 5521.7, "8-bit": 5629.1, "4-bit": 5568.4, "batch1": 5537, "batch8": 6362}
decode  = {"bf16": 162.8,  "8-bit": 257.7,  "4-bit": 349.3,  "batch1": 351.7, "batch8": 121.9}

print("量化 4-bit / bf16")
print(f"  prefill {prefill['4-bit']/prefill['bf16']:.3f}×   decode {decode['4-bit']/decode['bf16']:.2f}×")
print("batch 1→8（单路 decode；系统吞吐另算）")
print(f"  prefill {prefill['batch8']/prefill['batch1']:.3f}×   单路 decode {decode['batch8']/decode['batch1']:.2f}×")
print()
print("→ 量化提速、batch 提吞吐，两句话都只在 decode 上成立。")
print("  把 prefill 和 decode 混进一张『速度表』，结论会相反。")


## 陷阱 2：没对齐上下文 —— 比的是 KV，不是引擎


In [ ]:
# 同一 0.6B、同一台机器，只改 num_ctx（benchmark 陷阱 2）
rows = [(1024, 0.671), (4096, 1.0), (40960, 5.6)]
print(f"{'num_ctx':>10} {'ollama ps SIZE':>16} {'相对 1024':>10}")
base = rows[0][1]
for ctx, gb in rows:
    print(f"{ctx:>10,} {gb:>14.3f} GB {gb/base:>9.1f}×")
print()
print("默认 40960 是官方按 VRAM 档位给的，不是『引擎开销』。")
print("比内存之前：显式 num_ctx（或 OLLAMA_CONTEXT_LENGTH）+ ollama ps 看 CONTEXT。")


## 陷阱 3：「同模型」不等于「同精度 / 同体积」


In [ ]:
mlx4, gguf4 = 0.336, 0.517
print(f"Qwen3-0.6B 的两种 4-bit：MLX {mlx4} GB  vs  GGUF Q4_K_M {gguf4} GB  （差 {(gguf4/mlx4-1)*100:.0f}%）")
print()
print("同 4-bit 档  decode：MLX 349 vs llama.cpp 283  → MLX +23.5%")
print("同 16-bit    decode：MLX 163 vs llama.cpp 150  → MLX  +8.8%")
print()
print("锁精度后优势从 23.5% 缩到 8.8%。剩下的 15 个百分点是『MLX 这份 4-bit 更小』，")
print("不是引擎魔法。比较引擎看同精度；比较『我会怎么配』才看各自推荐档。")


## 陷阱 4：冷启动 vs 热态（前缀缓存）


In [ ]:
print("Ollama 同一 prompt：首次 TTFT 868 ms → 第二次 23 ms  （约 1/38）")
print("MLX server：        首次 172 ms → 热态 88 ms")
print()
print("多轮对话 / RAG / 固定 system prompt 都会命中前缀缓存。")
print("做 TTFT 对比：在 prompt 开头插变化的 nonce（放结尾，前缀仍命中）。")
print("做产品体验：反而应该尽量命中 —— 那是特性不是作弊。分清你在测哪一件。")


## 陷阱 5：软件版本与默认值在漂移


In [ ]:
print("本目录数字绑定的版本（不写进报告 = 半年后无法复核）：")
print("  llama.cpp  v0.4.0  build 10809   2026-09-11")
print("  Ollama     0.33.3                 2026-09-11")
print("  mlx-lm     0.31.3 / mlx 0.32.2    2026-09-12")
print()
print("已发生的破坏性变迁：")
print("  llama.cpp  bXXXX → 语义化 v0.4.0；--no-mmap 被 -lm/--load-mode 取代")
print("  Ollama     默认 ctx 按 VRAM 档位，与文档 4k/32k/256k 不完全一致")
print("             —— 0.6B 实测默认 40960，9B 默认 262144")
print()
print("任何 tok/s、GB、ms 都必须连同：机型 / 带宽 / 软件版本 / 模型 / 量化 / p长度 / n长度 / batch / ctx / 冷热。")


## 收尾：两行数字有没有资格放进同一张表


In [ ]:
def comparable(a: dict, b: dict) -> list[str]:
    keys = ("phase", "model", "quant", "weight_gb", "ctx", "prompt_n", "gen_n", "batch", "cold", "runtime_ver")
    bad = []
    labels = {
        "phase": "阶段（prefill/decode）",
        "model": "模型",
        "quant": "量化口径（不要混 MLX-4 与 Q4_K_M 当『同 4-bit』）",
        "weight_gb": "权重体积（同精度才比引擎）",
        "ctx": "上下文长度",
        "prompt_n": "prompt token 数",
        "gen_n": "生成长度",
        "batch": "batch",
        "cold": "冷/热态",
        "runtime_ver": "运行时+版本",
    }
    for k in keys:
        if a.get(k) != b.get(k):
            bad.append(f"{labels[k]}: {a.get(k)!r} vs {b.get(k)!r}")
    return bad


# 反例：看起来都是「Qwen3-0.6B 4-bit 速度」
left = dict(phase="decode", model="Qwen3-0.6B", quant="MLX-4", weight_gb=0.336,
            ctx=4096, prompt_n=512, gen_n=128, batch=1, cold=False, runtime_ver="mlx-lm 0.31.3")
right = dict(phase="decode", model="Qwen3-0.6B", quant="Q4_K_M", weight_gb=0.517,
             ctx=40960, prompt_n=168, gen_n=64, batch=1, cold=True, runtime_ver="Ollama 0.33.3")

issues = comparable(left, right)
print("这两行能不能比？")
if not issues:
    print("  可以。")
else:
    print(f"  不能，{len(issues)} 处没锁：")
    for line in issues:
        print("   -", line)

print()
print("报告只写三件套（缺一个都会被误读）：")
print("  1. prefill tok/s @ 指定 prompt 长度")
print("  2. decode tok/s")
print("  3. 峰值内存 @ 指定上下文")
